# Bulkhead: sandbox and permission layer Colab notebook

This notebook runs Bulkhead (`src/bulkhead/`), the layer that decides whether each of Steadyhand's tool calls may run (ALLOW, DENY, or ASK, with taint) and runs exec, filesystem, and network tools only inside an OS sandbox. The threat model is `Bulkhead/THREAT_MODEL.md`, and the eight-part plan is `Bulkhead/0N-*.md`. The agent loop itself is in `notebooks/02_steadyhand.ipynb`, and Toolsmith in `notebooks/01_toolsmith.ipynb`.

Run the cells top to bottom on a fresh Colab T4 or L4 runtime:

1. **Setup.** The same four cells as the other notebooks: they pull the repository, mount Drive, load `HF_TOKEN` and `LOCAL_KEY`, install the project, and detect the GPU, vCPUs, and RAM.
2. **Model runtime.** Starts restoring or building llama.cpp and copying the GGUF for the model in `STEADYHAND_PHASE`, in the background. Only the live AgentDojo sweep needs it; skip it, and the live section, for a CPU-only session.
3. **Capability probe (part 1, CPU).** Measures which sandbox tier this runtime can enforce: T2, T1, or T0, which is NOT a sandbox.
4. **Attack suite (part 7, CPU).** Runs the attack corpus on every profile the probe proved, then the usability benchmark, and writes `RESULTS.md`.
5. **AgentDojo CPU gate (part 8, CPU).** Installs the pinned AgentDojo and runs the adapter, runner, and metric tests against it. It must pass before the live sweep.
6. **Live AgentDojo sweep (part 8).** Starts one llama-server and runs `baseline`, `policy`, and `policy_taint` on the same cases, a smoke subset first.
7. **Stop** before switching models, and **cleanup** at the end.

Records go to `MyDrive/toolsmith/records/<gpu>/bulkhead/` (`t4` or `l4`): the probe report, attack, usability, and AgentDojo runs, and `RESULTS.md`. Every runner resumes from those records.

**Settings** are environment variables:
- Probe and attacks: `BULKHEAD_CGROUP_PARENT`, `BULKHEAD_REQUIRE_TIER`, `BULKHEAD_DESTRUCTIVE`, `BULKHEAD_OVERHEAD_SAMPLES`.
- AgentDojo: `BULKHEAD_AGENTDOJO_SMOKE`, `BULKHEAD_AGENTDOJO_CASES`, `BULKHEAD_AGENTDOJO_SUITES`, `BULKHEAD_AGENTDOJO_MAX_TOKENS`.
- Model: `STEADYHAND_PHASE`.

Put `HF_TOKEN` and `LOCAL_KEY` in `MyDrive/toolsmith/.env` (or `.env.txt`) or in Colab Secrets, never in a cell.

**Shared cells:** the setup cells match the Toolsmith notebook's, and the runtime, server, stop, and cleanup cells match the Steadyhand notebook's. `tests/test_notebook_03.py` keeps them identical, so change every copy together.

**After a disconnect:** reconnect and run all cells from the top. Builds and models come back from Drive. The attack suite, usability benchmark, and AgentDojo runs restore their records from Drive and run only what is missing.

In [ ]:
# Standard-library imports used throughout, and a first look at the GPU and Python version.
# Runs before anything else; it must work even on a CPU runtime.
import json
import os
import platform
import shlex
import shutil
import socket
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

# No check=True: on a CPU runtime the hardware cell explains how to switch instead.
gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True,
    text=True,
) if shutil.which('nvidia-smi') else None
gpu_info = gpu_result.stdout.strip() if gpu_result is not None and gpu_result.returncode == 0 else 'No NVIDIA GPU visible'
print(gpu_info)
print(f'Python {platform.python_version()}')

In [ ]:
# Clone or update the repository, mount Google Drive, create the shared cache folders, and load
# HF_TOKEN and LOCAL_KEY from MyDrive/toolsmith/.env or Colab Secrets (never from a cell).
from google.colab import drive, userdata

REPO_URL = 'https://github.com/shreyasnarahari/bare-metal-agent'
PROJECT_DIR = Path('/content/bare-metal-agent')
if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main'],
        check=True,
    )
elif PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} exists but is not a bare-metal-agent git checkout.'
    )
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
if not (PROJECT_DIR / 'pyproject.toml').is_file():
    raise FileNotFoundError(f'Expected {PROJECT_DIR / "pyproject.toml"} after repository setup.')
src_dir = PROJECT_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

drive.mount('/content/drive')
from toolsmith.provision import read_env_file

DRIVE_ROOT = Path('/content/drive/MyDrive/toolsmith')
DRIVE_BUILD_CACHE = DRIVE_ROOT / 'builds'
DRIVE_MODEL_CACHE = DRIVE_ROOT / 'models'
# Builds (keyed by compute capability) and models are shared by every GPU; records are per GPU.
for directory in (DRIVE_BUILD_CACHE, DRIVE_MODEL_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

env_path = next(
    (
        candidate
        for candidate in (DRIVE_ROOT / '.env', DRIVE_ROOT / '.env.txt')
        if candidate.is_file()
    ),
    DRIVE_ROOT / '.env',
)
env_values = read_env_file(env_path)
def load_secret(name):
    value = env_values.get(name, '').strip()
    if value:
        return value, env_path
    return userdata.get(name), 'Colab Secrets'

HF_TOKEN, hf_source = load_secret('HF_TOKEN')
LOCAL_KEY, local_source = load_secret('LOCAL_KEY')
if not HF_TOKEN or not LOCAL_KEY:
    raise ValueError(f'HF_TOKEN and LOCAL_KEY must be set in {env_path} or Colab Secrets.')
print(f'HF_TOKEN loaded from {hf_source}; LOCAL_KEY loaded from {local_source}')

In [ ]:
# Install this repository (editable, with the Colab extras) into the notebook's Python.
get_ipython().run_line_magic(
    'pip',
    f"install -q -e {shlex.quote(str(PROJECT_DIR) + '[colab]')}",
)

In [ ]:
# Detect the GPU, vCPUs, RAM, and disk once; every later size (build jobs, slots, pools) comes from
# HARDWARE. Also sets the per-GPU records folder, the background-job runner, and the record stamp.
import os
import uuid
from functools import partial
from pathlib import Path

from toolsmith.background import BackgroundJobs
from toolsmith.drive import ensure_drive, sync_drive
from toolsmith.hardware import detect_hardware, require_supported, runtime_stamp

# Every build, server, and pool size below comes from what this runtime has (T4, L4, or other).
HARDWARE = detect_hardware()
print(HARDWARE.summary())
require_supported(HARDWARE)
GPU_FAMILY = HARDWARE.family
CPU_COUNT = HARDWARE.cpu_count
# A T4 run and an L4 run never share a checkpoint, report, or JSONL file.
DRIVE_RECORDS = DRIVE_ROOT / 'records' / HARDWARE.records_slug
DRIVE_RECORDS.mkdir(parents=True, exist_ok=True)
print(f'Records for this GPU: {DRIVE_RECORDS}')
# Network calls give up instead of hanging: git aborts a transfer under 1 KB/s for a minute, and
# Hugging Face requests time out (read when huggingface_hub is first imported, after this cell).
for name, value in {
    'GIT_HTTP_LOW_SPEED_LIMIT': '1000', 'GIT_HTTP_LOW_SPEED_TIME': '60',
    'HF_HUB_DOWNLOAD_TIMEOUT': '60', 'HF_HUB_ETAG_TIMEOUT': '30',
}.items():
    os.environ.setdefault(name, value)
# Downloads, builds, and Drive copies run here while the GPU works; each job logs to its own file.
if 'BACKGROUND' not in globals():
    BACKGROUND = BackgroundJobs(Path(os.environ.get('TOOLSMITH_BACKGROUND_LOGS', '/content/background-logs')))
# Added to every Toolsmith record: the GPU, how busy the machine was, and this kernel's session, so
# records rewritten by a rerun after a disconnect can be told apart from the first run's.
if 'SESSION_ID' not in globals():
    SESSION_ID = uuid.uuid4().hex[:12]
RUNTIME_STAMP = partial(runtime_stamp, HARDWARE, BACKGROUND, session_id=SESSION_ID)

## Model runtime (for the live AgentDojo sweep)

This is the Steadyhand notebook's runtime cell, unchanged: `STEADYHAND_PHASE` (`qwen35/wide`, `qwen35/tall`, `qwen3/wide`, `qwen3/tall`) picks the model and server shape, and builds and weights come from the shared Drive cache. It only starts background jobs and returns, so the CPU sections below run while the model downloads and llama.cpp builds; the server cell in the live section waits for whatever is left.

One GGUF is resident at a time. To switch models, run the stop cell, change `STEADYHAND_PHASE`, and rerun this cell and the live section.

In [ ]:
# Start provisioning the model STEADYHAND_PHASE names (llama.cpp build, GGUF, renderer) as
# background jobs and return at once; the server cell collects them after the CPU gate.
from toolsmith.prompting import ChatPromptRenderer
from toolsmith.protocols import QWEN3_PROFILE, QWEN35_PROFILE
from toolsmith.provision import (
    LlamaCppRuntime,
    cache_dir_for,
    ensure_gguf,
    ensure_llama_server,
    q4_k_m_language_file,
    resolve_gguf,
)

STEADYHAND_MODELS = {
    'qwen3': {'profile': QWEN3_PROFILE, 'llama_cpp_ref': 'b5401',
              'repo': 'Qwen/Qwen3-8B-GGUF', 'filename': 'Qwen3-8B-Q4_K_M.gguf'},
    'qwen35': {'profile': QWEN35_PROFILE, 'llama_cpp_ref': 'b8233',
               'repo': 'unsloth/Qwen3.5-9B-GGUF', 'filename': None},
}
steadyhand_profile_name = os.environ.get('STEADYHAND_PHASE', 'qwen35/wide').split('/')[0]
if steadyhand_profile_name not in STEADYHAND_MODELS:
    raise ValueError(f'STEADYHAND_PHASE must start with one of {sorted(STEADYHAND_MODELS)}')
steadyhand_model_spec = STEADYHAND_MODELS[steadyhand_profile_name]
if globals().get('STEADYHAND_ACTIVE'):
    raise RuntimeError('A Steadyhand llama-server is running; run the stop cell before switching models')

GPU_NAME = HARDWARE.gpu_name
GPU_COMPUTE_CAP = HARDWARE.compute_cap
steadyhand_ref = steadyhand_model_spec['llama_cpp_ref']
steadyhand_llama_runtime = LlamaCppRuntime(
    ref=steadyhand_ref,
    source=Path(f'/content/llama.cpp-{steadyhand_ref}-source'),
    build=Path(f'/content/llama.cpp-{steadyhand_ref}-build'),
    cache=cache_dir_for(DRIVE_BUILD_CACHE, steadyhand_ref, GPU_COMPUTE_CAP),
    compute_cap=GPU_COMPUTE_CAP,
    staging=Path(f'/content/llama.cpp-{steadyhand_ref}-staging'),
)

def provision_steadyhand_gguf(spec, profile_name):
    if spec['filename']:
        gguf = resolve_gguf(spec['repo'], token=HF_TOKEN, filename=spec['filename'])
        drive_name = gguf.filename
    else:
        gguf = resolve_gguf(spec['repo'], token=HF_TOKEN, select=q4_k_m_language_file)
        drive_name = f'{gguf.revision[:12]}-{Path(gguf.filename).name}'
    # The same Drive names the Toolsmith notebook uses, so both notebooks share one cached copy.
    local = ensure_gguf(
        gguf,
        local_path=Path('/content') / f'{profile_name}-{Path(gguf.filename).name}',
        drive_path=DRIVE_MODEL_CACHE / drive_name,
        token=HF_TOKEN,
        cache_dir=f'/content/gguf-download-{profile_name}',
        defer=BACKGROUND.submit,
    )
    return gguf, local

# Started here and collected by the server cell, so the CPU gate below runs while the weights
# download or copy from Drive and llama.cpp restores or builds.
BACKGROUND.submit(
    f'steadyhand-llama.cpp-{steadyhand_ref}', ensure_llama_server, steadyhand_llama_runtime,
    jobs=HARDWARE.build_jobs(), defer=BACKGROUND.submit,
)
BACKGROUND.submit(f'steadyhand-gguf-{steadyhand_profile_name}', provision_steadyhand_gguf, steadyhand_model_spec, steadyhand_profile_name)
BACKGROUND.submit(
    f'steadyhand-renderer-{steadyhand_profile_name}', ChatPromptRenderer.from_pretrained,
    steadyhand_model_spec['profile'].tokenizer, token=HF_TOKEN,
)
STEADYHAND_PROVISIONING = {
    'profile_name': steadyhand_profile_name,
    'profile': steadyhand_model_spec['profile'],
    'llama_cpp_ref': steadyhand_ref,
    'build': steadyhand_llama_runtime.build,
    'gpu': GPU_NAME,
    'compute_cap': GPU_COMPUTE_CAP,
}
STEADYHAND_RUNTIME = None

def collect_steadyhand_runtime():
    provisioning = STEADYHAND_PROVISIONING
    name, ref = provisioning['profile_name'], provisioning['llama_cpp_ref']
    server_bin = BACKGROUND.wait(f'steadyhand-llama.cpp-{ref}')
    gguf, local_model = BACKGROUND.wait(f'steadyhand-gguf-{name}')
    return {
        **provisioning,
        'server_bin': server_bin,
        'model': local_model,
        'model_repo': gguf.repo,
        'model_file': gguf.filename,
        'model_revision': gguf.revision,
        'model_size': gguf.size,
        'renderer': BACKGROUND.wait(f'steadyhand-renderer-{name}'),
    }

print(f'Provisioning {steadyhand_profile_name} on llama.cpp {steadyhand_ref} in the background: {BACKGROUND.status()}')

## Bulkhead capability probe (part 1)

This cell runs `bulkhead.probe` on CPU while the runtime provisions. It tries real operations in temporary directories (bubblewrap namespaces, seccomp, `no_new_privs`, Landlock, rlimits, process cleanup, cgroup v2) and selects the strongest tier this runtime proves. The GPU type plays no part in the result.

The report goes to `MyDrive/toolsmith/records/<gpu>/bulkhead/probe.colab.json`, and a `T0` result is printed as **NOT a sandbox**. The probe installs nothing, changes no setting, and reads no secret. Being root inside the Colab container is not treated as permission to mount or to write cgroups: the cgroup checks run only under a parent named in `BULKHEAD_CGROUP_PARENT`. Set `BULKHEAD_REQUIRE_TIER` (`T1` or `T2`) to make the cell fail unless that tier is proven; it never falls back to a lower one. Part 1 runs no tool in any tier.

In [ ]:
# Bulkhead part 1: probe which sandbox tier this runtime can enforce (CPU only; the GPU type is irrelevant).
# Real operations in temporary directories; installs nothing, changes no setting, reads no secret.
import os
from pathlib import Path

from bulkhead.probe import render_report, run_probe, write_report

BULKHEAD_PROBE = run_probe(cgroup_parent=os.environ.get('BULKHEAD_CGROUP_PARENT') or None)
bulkhead_probe_path = write_report(
    BULKHEAD_PROBE, Path(DRIVE_RECORDS) / 'bulkhead' / 'probe.colab.json', secrets=(HF_TOKEN, LOCAL_KEY),
)
print(render_report(BULKHEAD_PROBE))
print(f'Probe report: {bulkhead_probe_path}')
bulkhead_required_tier = os.environ.get('BULKHEAD_REQUIRE_TIER')
if bulkhead_required_tier:
    BULKHEAD_PROBE.require(bulkhead_required_tier)  # raises UnsupportedTierError; never a lower tier

## Bulkhead attack suite and results (part 7)

This cell runs the versioned attack corpus (`evaluation/data/bulkhead_attacks_v1.json`: 59 attacks and 6 positive controls) on every execution profile the probe above proves on this runtime, then the approval-usability benchmark, and writes `results/bulkhead/RESULTS.md` from those records only. It needs no GPU.

- **Recorded calls first:** a run is identified by the dataset, code, configuration, and probe. Its records are mirrored to `MyDrive/toolsmith/records/<gpu>/bulkhead/attacks/`, and a fresh runtime restores them and runs only the units that are missing.
- **Unsupported profiles are untested, not skipped:** a profile this kernel cannot build (no bubblewrap, no delegated cgroup) gets an `untested` record per attack, with the probe's reason. A tier is never silently replaced by a lower one.
- **Destructive attacks are opt-in:** fork, memory, CPU, and disk bombs run only with `BULKHEAD_DESTRUCTIVE=1` *and* a delegated parent in `BULKHEAD_CGROUP_PARENT`.
- **Overhead is a separate serial measurement:** set `BULKHEAD_OVERHEAD_SAMPLES` (for example `1000`) on an otherwise idle runtime. Colab timings are not comparable with the development host's.
- **Escapes fail the cell:** any `escaped` verdict raises. It is a release blocker, kept as a permanent regression.

In [ ]:
# Bulkhead part 7: the attack corpus on every proven profile, the usability benchmark, and RESULTS.md (CPU only).
import os
import shutil
import tempfile
from pathlib import Path

from bulkhead.cgroups import configured_parent
from bulkhead.eval.attacks import discover_targets, load_dataset, run_suite, summarize
from bulkhead.eval.report import write_report
from bulkhead.eval.usability import run_usability

bulkhead_results = Path(PROJECT_DIR) / 'results' / 'bulkhead'
bulkhead_drive = Path(DRIVE_RECORDS) / 'bulkhead'
bulkhead_parent = configured_parent()
bulkhead_work = Path(tempfile.mkdtemp(prefix='bulkhead-attack-work-'))
try:
    bulkhead_targets = discover_targets(bulkhead_work, report=BULKHEAD_PROBE, cgroup_parent=bulkhead_parent)
    for target in bulkhead_targets:
        print(f'{target.profile:<22} ' + ('available' if target.available else f'untested: {target.reason}'))
    bulkhead_attack_run = run_suite(
        load_dataset(), bulkhead_targets, results_dir=bulkhead_results / 'attacks', probe=BULKHEAD_PROBE,
        destructive=os.environ.get('BULKHEAD_DESTRUCTIVE') == '1' and bulkhead_parent is not None,
        workers=HARDWARE.cpu_workers(), mirror_dir=bulkhead_drive / 'attacks',
    )
    for profile, counts in summarize(bulkhead_attack_run.records).items():
        print(f'  {profile:<22} ' + ', '.join(f'{label} {count}' for label, count in sorted(counts.items())))
    bulkhead_overhead_runs = []
    if int(os.environ.get('BULKHEAD_OVERHEAD_SAMPLES') or 0):
        from bulkhead.eval.overhead import run_overhead

        bulkhead_overhead_runs.append(run_overhead(
            bulkhead_targets, samples=int(os.environ['BULKHEAD_OVERHEAD_SAMPLES']), probe=BULKHEAD_PROBE,
            results_dir=bulkhead_results / 'overhead',
        ))
    bulkhead_usability_run = run_usability(results_dir=bulkhead_results / 'usability')
    bulkhead_report = write_report(
        bulkhead_results, attack_runs=[bulkhead_attack_run.directory], overhead_runs=bulkhead_overhead_runs,
        usability_runs=[bulkhead_usability_run],
    )
finally:
    shutil.rmtree(bulkhead_work, ignore_errors=True)
bulkhead_drive.mkdir(parents=True, exist_ok=True)
shutil.copy(bulkhead_report, bulkhead_drive / 'RESULTS.md')
print(f'Report: {bulkhead_report} (copied to {bulkhead_drive / "RESULTS.md"})')
bulkhead_escapes = sorted({(r['attack_id'], r['profile']) for r in bulkhead_attack_run.latest().values() if r.get('verdict') == 'escaped'})
if bulkhead_escapes or bulkhead_attack_run.interrupted:
    raise RuntimeError(f'Bulkhead release blockers: escapes {bulkhead_escapes}, interrupted {bulkhead_attack_run.interrupted}')

## Bulkhead on AgentDojo (part 8)

Two cells measure whether Bulkhead changes an agent's behavior under prompt injection: the CPU gate here, which needs no model, and the paired sweep in the live section below, which runs on the resident model.

- **Pinned, evaluation-only dependency:** the gate installs exactly `agentdojo 0.1.35` at commit `a75aba76` (`bulkhead.eval.agentdojo.AGENTDOJO_REQUIREMENT`). It is not a project dependency, and the adapter refuses any other version.
- **CPU gate first:** the adapter, runner, and metric tests run against the real package: a local three-task fixture scored identically by the real AgentDojo and the test stand-in, and the real v1.2.2 suites planned, with their withheld tools listed by JSON Pointer and their output-trust declarations checked with canaries. No live score can be inferred from it. The part 7 attack suite above runs before any claim about the combined system.
- **Three paired configurations on the same case ids:** `baseline` (Steadyhand alone), `policy` (the static per-suite task policy), and `policy_taint` (the same policy, plus an ASK for every sensitive sink once the episode has read untrusted output). Model, seed, temperature, tools, injections, and the empty fault schedule are identical; the model is never reloaded for a policy change. ASK uses `deny_all`, and nothing is auto-approved.
- **Smoke subset, then the selection:** `BULKHEAD_AGENTDOJO_SMOKE` (default 6 cases, all three configurations) runs first; read its journals before the long run. `BULKHEAD_AGENTDOJO_CASES` is a count for a deterministic subset (seed 0), `all`, or `0` to stop after the smoke. A subset is reported as a partial run with its exact ids.
- **Records:** written locally to `/content/bulkhead/agentdojo/<run_id>/`, with records and journals copied to `MyDrive/toolsmith/records/<gpu>/bulkhead/agentdojo/`. A fresh runtime resumes from Drive. A changed model, policy, or approval mode is a new run id. Infrastructure aborts are recorded, excluded from every denominator, and rerun at most twice; after repeated aborts the server is restarted once.

In [ ]:
# Bulkhead part 8 CPU gate: install the pinned AgentDojo (evaluation only), run the adapter, runner,
# and metric tests against it, and record the result under this GPU's records. No model and no GPU.
import importlib
import subprocess
import sys

from bulkhead.eval.agentdojo import AGENTDOJO_REQUIREMENT, AGENTDOJO_VERSION, installed_pin, require_pinned
from toolsmith.evaluation.runner import atomic_write_json

if (installed_pin() or {}).get('version') != AGENTDOJO_VERSION:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', AGENTDOJO_REQUIREMENT], check=True)
    importlib.invalidate_caches()
bulkhead_agentdojo_pin = require_pinned()
bulkhead_gate_started = time.perf_counter()
bulkhead_gate_run = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', '-W', 'ignore::DeprecationWarning',
     'tests/test_bulkhead_agentdojo.py', 'tests/test_bulkhead_eval_runner.py', 'tests/test_bulkhead_eval_metrics.py'],
    cwd=PROJECT_DIR, capture_output=True, text=True,
)
bulkhead_gate_tail = (bulkhead_gate_run.stdout.strip().splitlines() or [''])[-1]
BULKHEAD_AGENTDOJO_GATE = {
    'passed': bulkhead_gate_run.returncode == 0 and 'skipped' not in bulkhead_gate_tail,
    'summary': bulkhead_gate_tail,
    'agentdojo': bulkhead_agentdojo_pin,
    'seconds': round(time.perf_counter() - bulkhead_gate_started, 1),
}
atomic_write_json(Path(DRIVE_RECORDS) / 'bulkhead' / 'agentdojo' / 'cpu_gate.json', BULKHEAD_AGENTDOJO_GATE)
print(json.dumps(BULKHEAD_AGENTDOJO_GATE, indent=2))
if not BULKHEAD_AGENTDOJO_GATE['passed']:
    print(bulkhead_gate_run.stdout[-4000:])
    raise RuntimeError('The Bulkhead AgentDojo CPU gate failed (or skipped tests): fix it before any live run')

## Live AgentDojo sweep (part 8)

The server cell is the Steadyhand notebook's, unchanged. It starts one llama-server in the best shape this GPU holds (KV cache and flash-attention ladder, slots shrunk on out-of-memory, decode self-test). The sweep cell then runs the paired AgentDojo configurations with one worker per slot, a smoke subset first, as described in the part 8 notes above. It restarts the server once if cases keep aborting on infrastructure.

In [ ]:
# Collect the provisioned runtime and start llama-server in the best shape this GPU holds (KV cache
# and flash-attention ladder, slots shrunk on OOM, decode self-test). Produces STEADYHAND_ACTIVE.
import subprocess
import time

from steadyhand import LiveSettings, build_llama_client
from steadyhand.eval.serving import (
    SELF_TEST_PROMPT,
    decode_speed,
    fit_server,
    kv_ladder,
    parse_nvidia_smi,
    preset_for,
    server_extra_args,
)
from toolsmith.backends import CompletionTruncatedError, LlamaCppClient, LlamaCppServer
from toolsmith.provision import free_port, server_environment

def stop_steadyhand_runtime():
    global STEADYHAND_ACTIVE
    active = globals().get('STEADYHAND_ACTIVE') or {}
    if active.get('client') is not None:
        active['client'].close()
    if active.get('server') is not None:
        active['server'].stop(timeout=30)
    STEADYHAND_ACTIVE = {}
    os.environ.pop('TOOLSMITH_BASE_URL', None)

def query_gpu():
    output = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.used,utilization.gpu',
         '--format=csv,noheader,nounits'],
        check=True, capture_output=True, text=True,
    ).stdout
    return parse_nvidia_smi(output)[0]

if not globals().get('STEADYHAND_PROVISIONING'):
    raise RuntimeError('Run the Steadyhand runtime cell first')
steadyhand_profile_name, steadyhand_shape = os.environ.get('STEADYHAND_PHASE', 'qwen35/wide').split('/')
if steadyhand_profile_name != STEADYHAND_PROVISIONING['profile_name']:
    raise RuntimeError(
        f"STEADYHAND_PHASE asks for {steadyhand_profile_name}, but the runtime cell provisioned "
        f"{STEADYHAND_PROVISIONING['profile_name']}; stop the server and rerun the runtime cell"
    )
# Waits for whatever the runtime cell's background jobs have not finished during the CPU gate.
if not globals().get('STEADYHAND_RUNTIME'):
    STEADYHAND_RUNTIME = collect_steadyhand_runtime()
steadyhand_runtime = STEADYHAND_RUNTIME

steadyhand_log = Path('/content/steadyhand-server.log')

def start_steadyhand_server(preset, extra_args, port):
    server = LlamaCppServer(
        executable=steadyhand_runtime['server_bin'], model=steadyhand_runtime['model'], api_key=LOCAL_KEY,
        log_path=steadyhand_log, port=port, context_size=preset.context_size,
        gpu_layers=99, parallel=preset.parallel, extra_args=extra_args,
    )
    probe = LlamaCppClient(server.base_url, LOCAL_KEY, timeout=30.0)
    try:
        server.start()
        server.wait_until_healthy(probe, timeout=900, poll_interval=2)
    except BaseException:
        server.stop(timeout=30)
        raise
    finally:
        probe.close()
    return server

def probe_decode_speed(server):
    # One short completion: a GPU-resident model decodes far above the floor, a CPU fallback does not.
    with LlamaCppClient(server.base_url, LOCAL_KEY, timeout=300.0) as client:
        started = time.perf_counter()
        try:
            response = client.complete(SELF_TEST_PROMPT, max_tokens=64, temperature=0.0, seed=0)
        except CompletionTruncatedError as error:
            response = error.response
        return decode_speed(response, time.perf_counter() - started)

def start_steadyhand_runtime():
    # Stops any Steadyhand server, then fits and starts this phase's. The sweep cell calls it
    # again if the server dies mid-sweep.
    global STEADYHAND_ACTIVE
    # One resident GGUF: a previous Steadyhand server is stopped before this one loads.
    stop_steadyhand_runtime()
    steadyhand_env = server_environment(steadyhand_runtime['build'])
    os.environ['LD_LIBRARY_PATH'] = steadyhand_env['LD_LIBRARY_PATH']
    steadyhand_help = subprocess.run(
        [str(steadyhand_runtime['server_bin']), '--help'], capture_output=True, text=True, env=steadyhand_env,
    ).stdout
    steadyhand_gpu = query_gpu()
    # A card that is neither a T4 nor an L4 borrows the preset rows for its VRAM.
    steadyhand_preset = preset_for(
        steadyhand_gpu['name'], steadyhand_profile_name, steadyhand_shape, vram_mib=steadyhand_gpu['memory_total_mib'],
    )
    steadyhand_port = free_port('127.0.0.1', 8090)
    # Tries q8_0 KV with flash attention, then f16 with it, then f16 without it. Running out of VRAM
    # costs one slot (same per-slot context); a crash or a slow self-test moves to the next setting.
    steadyhand_fit = fit_server(
        steadyhand_preset, kv_ladder(steadyhand_help),
        start=lambda preset, extra_args: start_steadyhand_server(preset, extra_args, steadyhand_port),
        extra_args_for=lambda rung: server_extra_args(
            steadyhand_help, cpu_count=HARDWARE.cpu_count, kv_cache=rung['kv_cache'], flash_attn=rung['flash_attn'],
        ),
        query_gpu=query_gpu,
        probe=probe_decode_speed,
    )
    steadyhand_server, steadyhand_preset = steadyhand_fit.server, steadyhand_fit.preset
    os.environ['TOOLSMITH_BASE_URL'] = steadyhand_server.base_url
    STEADYHAND_ACTIVE = {
        'server': steadyhand_server,
        'preset': steadyhand_preset,
        'profile': steadyhand_runtime['profile'],
        'renderer': steadyhand_runtime['renderer'],
        'gpu': steadyhand_fit.gpu,
        'extra_args': steadyhand_fit.extra_args,
        'kv_cache': steadyhand_fit.rung['kv_cache'],
        'flash_attn': steadyhand_fit.rung['flash_attn'],
        'decode_tps': steadyhand_fit.decode_tps,
        'fit_attempts': steadyhand_fit.attempts,
        # The pool admits a little more than parallel so verify or backoff never queues a slot.
        'client': build_llama_client(
            LiveSettings.from_env(api_key=LOCAL_KEY),
            connections=steadyhand_preset.parallel + 2, timeout=900.0,
        ),
    }
    return STEADYHAND_ACTIVE

steadyhand_active = start_steadyhand_runtime()
steadyhand_slot_lines = [line for line in steadyhand_log.read_text(errors='replace').splitlines() if 'n_ctx' in line][:4]
print(json.dumps({
    **steadyhand_active['preset'].as_dict(), 'gpu': steadyhand_active['gpu'], 'extra_args': steadyhand_active['extra_args'],
    'kv_cache': steadyhand_active['kv_cache'], 'flash_attn': steadyhand_active['flash_attn'],
    'self_test_decode_tps': steadyhand_active['decode_tps'], 'attempts': steadyhand_active['fit_attempts'],
}, indent=2))
print('Server log context lines (confirm the per-slot split):', *steadyhand_slot_lines, sep='\n  ')

In [ ]:
# Bulkhead part 8: the paired AgentDojo sweep (baseline, policy, policy + taint) on the resident model.
# A smoke subset runs first, then the selection; both resume from Drive. Writes RESULTS.md from the records.
from bulkhead.eval.agentdojo import agentdojo_api
from bulkhead.eval.report import write_report
from bulkhead.eval.runner import RunInterrupted, make_spec, plan_cases, replay_run, run_agentdojo, select_cases, write_summary
from steadyhand import SteadyhandClient, worker_service
from steadyhand.controller import Budgets
from steadyhand.journal import JournalCopier
from toolsmith.artifacts import copy_file_atomic

if not globals().get('STEADYHAND_ACTIVE'):
    raise RuntimeError('Run the model runtime and server cells first')
if not (globals().get('BULKHEAD_AGENTDOJO_GATE') or {}).get('passed'):
    raise RuntimeError('Run the Bulkhead AgentDojo CPU gate first')
bulkhead_state = {'active': STEADYHAND_ACTIVE}
bulkhead_max_tokens = int(os.environ.get('BULKHEAD_AGENTDOJO_MAX_TOKENS', '512'))
bulkhead_budgets = Budgets(max_steps=15, max_prompt_tokens=400_000, max_completion_tokens=16_000)
bulkhead_suites = [s for s in os.environ.get('BULKHEAD_AGENTDOJO_SUITES', '').split(',') if s] or None
bulkhead_plan = plan_cases(agentdojo_api(), suites=bulkhead_suites)
bulkhead_local = Path(os.environ.get('BULKHEAD_AGENTDOJO_LOCAL', '/content/bulkhead/agentdojo'))
bulkhead_drive_runs = Path(DRIVE_RECORDS) / 'bulkhead' / 'agentdojo'
print(f'{len(bulkhead_plan.cases)} eligible cases, {len(bulkhead_plan.ineligible)} ineligible; withheld: '
      + ', '.join(f'{name}.{row.tool} ({row.pointer})' for name, a in bulkhead_plan.adapters.items() for row in a.withheld))

def bulkhead_client(case):
    # One fresh client per episode on the shared llama-server; the same settings in every configuration.
    active = bulkhead_state['active']
    return SteadyhandClient(
        worker_service(active['client'], active['renderer'], active['profile']),
        max_tokens=bulkhead_max_tokens, seed=0, slot_context=active['preset'].per_slot_context,
    )

def bulkhead_server_shape():
    active = bulkhead_state['active']
    return {
        **active['preset'].as_dict(), 'gpu': active['gpu']['name'], 'kv_cache_type': active['kv_cache'],
        'flash_attn': active['flash_attn'], 'extra_args': active['extra_args'],
        # The GGUF (repository, file, revision: the quantization) and the llama.cpp tag that serve it.
        **{key: STEADYHAND_RUNTIME.get(key) for key in ('llama_cpp_ref', 'model_repo', 'model_file', 'model_revision')},
    }

def bulkhead_run(cases, label):
    spec = make_spec(bulkhead_plan, cases, label=label, kind='live', model=bulkhead_client(cases[0]).settings(),
                     server=bulkhead_server_shape(), budgets=bulkhead_budgets)
    for attempt in (1, 2):
        try:
            with JournalCopier(bulkhead_drive_runs / spec.run_id / 'journals') as copier:
                directory = run_agentdojo(
                    spec, bulkhead_plan, bulkhead_client, results_dir=bulkhead_local,
                    workers=bulkhead_state['active']['preset'].parallel, budgets=bulkhead_budgets,
                    mirror_dir=bulkhead_drive_runs, copier=copier,
                )
            break
        except RunInterrupted as error:
            # Cases kept aborting on infrastructure: restart the server once and resume.
            if attempt == 2:
                raise
            print(f'{error}\nRestarting llama-server and resuming.', flush=True)
            bulkhead_state['active'] = start_steadyhand_runtime()
    summary = write_summary(directory, spec)
    replay = replay_run(directory, bulkhead_plan, mirror=bulkhead_drive_runs / spec.run_id)
    for name in ('summary.json', 'replay.json', 'eligibility.json'):
        copy_file_atomic(directory / name, bulkhead_drive_runs / spec.run_id / name)
    for configuration, s in summary['suites']['all'].items():
        print(f"  {label:<6} {configuration:<13} clean {s['utility_clean']['count']}/{s['utility_clean']['total']}  "
              f"attacked {s['utility_attack']['count']}/{s['utility_attack']['total']}  ASR {s['attack_success']['count']}/{s['attack_success']['total']}  "
              f"ASK rate {s['ask_rate']['count']}/{s['ask_rate']['total']}  not scored {s['not_scored']}")
    print(f"  replay: {replay['exact']}/{replay['episodes']} exact; records: {directory}")
    return directory

ensure_drive()
bulkhead_started = time.perf_counter()
bulkhead_runs = [bulkhead_run(select_cases(bulkhead_plan, count=int(os.environ.get('BULKHEAD_AGENTDOJO_SMOKE', '6')), seed=0), 'smoke')]
bulkhead_selection = os.environ.get('BULKHEAD_AGENTDOJO_CASES', 'all')
if bulkhead_selection != '0':
    bulkhead_count = None if bulkhead_selection == 'all' else int(bulkhead_selection)
    bulkhead_runs.append(bulkhead_run(select_cases(bulkhead_plan, count=bulkhead_count, seed=0), 'sweep'))
print(f'AgentDojo runs finished in {time.perf_counter() - bulkhead_started:.0f}s')
bulkhead_report = write_report(Path(PROJECT_DIR) / 'results' / 'bulkhead', agentdojo_runs=bulkhead_runs)
copy_file_atomic(bulkhead_report, Path(DRIVE_RECORDS) / 'bulkhead' / 'RESULTS.md')
print(f'Report: {bulkhead_report}')
sync_drive(BACKGROUND, stage='bulkhead-agentdojo')  # push this cell's results to Google's servers

In [ ]:
# Stop the Steadyhand server (run before switching STEADYHAND_PHASE to another model).
if 'stop_steadyhand_runtime' in globals():
    stop_steadyhand_runtime()
print('Steadyhand llama-server stopped. Start the next phase or model, or disconnect once the Drive copies are done.')

In [ ]:
# Stop the server, let Drive copies finish, stop background jobs, and flush Drive. Keep last.
if 'stop_steadyhand_runtime' in globals():
    stop_steadyhand_runtime()
if 'BACKGROUND' in globals():
    # Let Drive write-backs finish (they are atomic, so an unfinished one is redone next runtime).
    unfinished = BACKGROUND.drain(timeout=float(os.environ.get('TOOLSMITH_CLEANUP_DRAIN_SECONDS', '600')))
    stopped = BACKGROUND.cancel_all()
    if unfinished:
        print(f'Background jobs not finished cleanly: {unfinished}; logs in {BACKGROUND.log_dir}')
    if stopped:
        print(f'Stopped background jobs: {stopped}')
    # Flush every pending Drive write before the runtime goes away (Drive uploads in the background).
    sync_drive(BACKGROUND, stage='cleanup', remount=False)
print('Steadyhand llama-server stopped; Drive flushed and unmounted. Record Colab units used, then delete or disconnect the runtime.')